# Exercise 05: Semantic Search

> **Chapter:** Ch05 - Semantic Search
> **Estimated time:** ~50 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

Semantic search is more than ranking by meaning. A real search box also lets users narrow
results by hard constraints (a release year, a minimum rating, a genre) and shows them which
refinements are worth making. In this exercise you turn the retrieval building blocks from the
chapter into a small movie search engine over 500 films, then extend it with two features every
production system needs but the chapter only gestures at: filtered search and faceted navigation.
You assemble the four pipelines, add a predicate filter, build genre and rating facets, and
finally reason about where faceting belongs in the pipeline.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] 💻 **C1: Assemble the four pipelines** - compose BM25, dense, fusion, and reranking into pipelines A to D
- [ ] 💻 **C2: Filtered search** - restrict a search to the movies that satisfy a predicate
- [ ] 💻 **C3: Faceted navigation** - summarise a result set by genre and by rating band
- [ ] ✏ **T1: Where facets belong** - decide how a chosen facet should re-enter the pipeline

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"05 - Semantic Search"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: you build a working
search engine, give it a filter and facets, and reason about a design choice the chapter leaves
open.

---

## The movie collection

You work with `movies-small`, the first 500 films of the Kaggle movie dataset, the same slice the
chapter demo uses. Each movie is a plain Python dict with a searchable `text` field (title,
overview, tagline, top cast, and genres) plus structured metadata you can filter and group on:

| Field | Type | Example |
|---|---|---|
| `id`, `title`, `year` | str, str, int | `tt0114709`, `Toy Story`, `1995` |
| `rating`, `runtime` | float, int | `7.7`, `81` |
| `genres` | list[str] | `["Animation", "Comedy", "Family"]` |
| `cast`, `tagline`, `text` | str | ... |
| `tokens` | list[str] | stopword-filtered terms, for BM25 |
| `emb_mini`, `emb_qwen` | np.ndarray | pre-computed sentence embeddings |

The two embedding matrices are **pre-computed and shipped with the exercise**, so you never encode
the 500 documents yourself. Only your live query is encoded, by a small model you load below. There
is no vector index: the collection is a plain list, and search is a linear scan over it. That keeps
the code readable and is fast enough for 500 films.

> **First run downloads models.** Encoding your query needs a sentence-transformer. The default
> `mini` setting downloads all-MiniLM-L6-v2 (about 90 MB) and the cross-encoder
> ms-marco-MiniLM-L-6-v2 (about 90 MB) on first use, then caches them. Switching `MODEL` to `qwen`
> downloads the stronger Qwen3-Embedding-0.6B (about 1.2 GB) instead; it is optional.

In [ ]:
# Standard setup - run this first
import sys, pathlib
from collections import Counter

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_here = pathlib.Path().resolve()
_p = _here
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))       # the shared/ root, for `from shared...`
sys.path.insert(0, str(_here))    # the notebook's own folder wins for `import tasks`

%load_ext autoreload
%autoreload 2

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

from shared.semantic_movies import load_movies, EMBEDDING_FIELD
from shared.display import display_md, print_table

movies = load_movies()
display_md(f"**Loaded {len(movies)} movies.** Example: "
           f"*{movies[0]['title']}* ({movies[0]['year']}), genres {movies[0]['genres']}.")

Pick the embedding model, then load it and the cross-encoder. The `encode_query` function turns a
query string into one normalised vector, so a dot product against the stored embeddings equals
cosine similarity. The cross-encoder scores a (query, movie) pair jointly and is used only to
rerank a short candidate list.

In [ ]:
MODEL = "mini"   # "mini" (fast, 90 MB) or "qwen" (stronger, 1.2 GB download)

if MODEL == "qwen":
    _encoder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    def encode_query(text: str) -> np.ndarray:
        """Encode one query into a normalised embedding (Qwen expects a query prompt)."""
        vec = _encoder.encode([text], prompt_name="query",
                              normalize_embeddings=True, show_progress_bar=False)[0]
        return np.asarray(vec, dtype=np.float32)
else:
    _encoder = SentenceTransformer("all-MiniLM-L6-v2")
    def encode_query(text: str) -> np.ndarray:
        """Encode one query into a normalised embedding."""
        vec = _encoder.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]
        return np.asarray(vec, dtype=np.float32)

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
display_md(f"**Model ready:** `{MODEL}` (field `{EMBEDDING_FIELD[MODEL]}`) + cross-encoder.")

## The engine you will complete

Your code lives in <a href="tasks.py">tasks.py</a>, a small `MovieSearch` class. The four retrieval
building blocks are **already written for you**, exactly as in the chapter demo:

- `bm25(query, candidates, top_k)` ranks a list of movies by BM25 on their tokens.
- `dense(query, candidates, top_k)` encodes the query and ranks by cosine on the chosen embedding.
- `reciprocal_rank_fusion(ranked_lists, k)` merges ranked lists using ranks, not raw scores.
- `cross_encoder_rerank(query, candidate_ids, top_k)` reorders a candidate list by joint relevance.

Each block returns a list of `(movie_id, score)` pairs. Every method takes an explicit
`candidates` list to score, which is the hook you use later for filtered search. Four methods are
left as stubs for you to implement: `run_pipeline`, `search`, `genre_facets`, and `rating_facets`.
Open <a href="tasks.py">tasks.py</a> next to this notebook, fill in the stubs, and re-run the verify
cells. Autoreload picks up each save, so you never restart the kernel.

Build the engine once. It reuses the movies, the query encoder, and the cross-encoder from the
setup cells.

In [ ]:
from tasks import MovieSearch

engine = MovieSearch(movies, encode_query,
                     model_key=EMBEDDING_FIELD[MODEL], cross_encoder=cross_encoder)
display_md("**Engine built.** Implement the stubs in `tasks.py`, then run the verify cells.")

## Assembling the four pipelines

> **How this works:** Implement `run_pipeline` in <a href="tasks.py">tasks.py</a> using the four
> provided blocks. Run the verify cell; it either passes or fails with a specific message. Write
> the code yourself or direct an AI from the method's docstring.

### 💻 Task C1 - Compose pipelines A to D

The chapter presents five pipeline designs. Here you build the first four (E targets collections too
large to scan, which is not a concern for 500 films). Each is a different composition of the same
blocks: A is BM25 alone, B reranks BM25 with the cross-encoder, C is dense retrieval alone, and D
fuses BM25 and dense with Reciprocal Rank Fusion and then reranks. Fill in `run_pipeline` so each
label produces the right composition, then run every pipeline on the same two queries below and
read how the ranking changes.

In [ ]:
# --- Verify C1 ---
# A exact-title query: BM25 keys on the literal tokens.
rA = engine.run_pipeline("GoldenEye", "A", engine.movies, top_k=5)
assert rA, "pipeline A returned nothing for 'GoldenEye'"
assert engine.id_to_movie[rA[0][0]]["title"] == "GoldenEye", \
    f"A: expected GoldenEye on top, got {engine.id_to_movie[rA[0][0]]['title']}"
scores = [s for _, s in rA]
assert scores == sorted(scores, reverse=True), "results must be sorted by descending score"

# A paraphrase query with no shared title words: dense retrieval bridges the gap.
toys = "animated toys that come to life"
rC = engine.run_pipeline(toys, "C", engine.movies, top_k=5)
assert engine.id_to_movie[rC[0][0]]["title"] == "Toy Story", \
    f"C: expected Toy Story on top, got {engine.id_to_movie[rC[0][0]]['title']}"

# B and D add a cross-encoder stage; Toy Story must survive into the top 5.
for p in ("B", "D"):
    r = engine.run_pipeline(toys, p, engine.movies, top_k=5)
    assert len(r) <= 5, f"{p}: at most top_k results"
    titles = [engine.id_to_movie[i]["title"] for i, _ in r]
    assert "Toy Story" in titles, f"{p}: expected Toy Story in the top 5, got {titles}"
    s = [sc for _, sc in r]
    assert s == sorted(s, reverse=True), f"{p}: results must be sorted by descending score"

print("✓ pipelines A to D assemble correctly!")

In [ ]:
# --- Explore C1 ---
# Run the same query through all four pipelines and compare the top result.
def top_of(query, k=1):
    rows = []
    for p in ("A", "B", "C", "D"):
        r = engine.run_pipeline(query, p, engine.movies, top_k=k)
        rows.append([p, ", ".join(engine.id_to_movie[i]["title"] for i, _ in r) or "-"])
    display_md(f"**Top result per pipeline for `{query}`:**")
    print_table(rows, headers=["Pipeline", f"Top {k}"])

top_of("a dark thriller about a serial killer")
# Which pipeline would you ship for a general movie search box, and why?

## Filtered search

A user rarely wants the whole catalogue. They want "a thriller from before 1995" or "something
rated above 8". The clean way to honour a hard constraint is to apply it **before** scoring: build
the candidate list from the movies that satisfy the constraint, then run the pipeline over only
those. A filter expressed this way can never leak a movie it should have excluded, and the ranking
is computed over exactly the right set.

> **How this works:** Implement `search` in <a href="tasks.py">tasks.py</a>. It takes a `predicate`
> (a function from a movie dict to `True`/`False`) and runs the chosen pipeline over just the
> matching movies. The verify cell passes you ready-made predicates.

### 💻 Task C2 - Restrict a search with a predicate

Complete `search` so that, when a predicate is given, only the movies for which it returns `True`
are scored. When the predicate is `None`, the whole collection is searched. Because every movie is
a dict, a predicate is a one-line lambda over its fields, for example `lambda m: m["year"] < 1995`
or `lambda m: "Comedy" in m["genres"]`.

In [ ]:
# --- Verify C2 ---
# A year filter: every hit must satisfy it, and none must slip through.
older = engine.search("a thrilling adventure", pipeline="C",
                       predicate=lambda m: m["year"] < 1995, top_k=10)
assert older, "expected hits in the pre-1995 subset"
assert all(engine.id_to_movie[i]["year"] < 1995 for i, _ in older), \
    "every hit must satisfy year < 1995"

# A rating filter: exactly 10 movies in the collection are rated above 8.
top_rated = engine.search("a dark crime story", pipeline="C",
                          predicate=lambda m: m["rating"] > 8, top_k=10)
assert all(engine.id_to_movie[i]["rating"] > 8 for i, _ in top_rated), \
    "every hit must have rating > 8"
assert len(top_rated) == 10, f"the rating>8 subset has 10 movies, got {len(top_rated)}"

# No predicate searches the whole collection.
everything = engine.search("comedy about friendship", pipeline="C", top_k=5)
assert len(everything) == 5, "an unfiltered search still returns top_k results"

print("✓ filtered search works!")

In [ ]:
# --- Explore C2 ---
# Watch the top result change when the same query is constrained.
def show(query, predicate=None, label=""):
    r = engine.search(query, pipeline="C", predicate=predicate, top_k=3)
    titles = [f"{engine.id_to_movie[i]['title']} "
              f"({engine.id_to_movie[i]['year']}, r{engine.id_to_movie[i]['rating']})"
              for i, _ in r]
    display_md(f"**{label or 'all movies'}:** {', '.join(titles)}")

show("an epic war story")
show("an epic war story", lambda m: m["rating"] > 8, "only rated > 8")
# Try a genre predicate, or combine two conditions with `and`.

## Faceted navigation

Good search interfaces do not only return a ranked list. They also show the user how the results
break down, so they can refine. An online shop shows "Brand (42), Colour (37), ..." next to the
results. Here you build the equivalent for movies: a genre breakdown and a rating breakdown of the
current result set. These counts are what a user interface would render as clickable filters.

> **How this works:** Implement `genre_facets` and `rating_facets` in <a href="tasks.py">tasks.py</a>.
> Both take a ranking (a list of `(movie_id, score)` pairs) and summarise it.

### 💻 Task C3 - Summarise results by genre and by rating

`genre_facets(results, top_n)` counts how often each genre appears across the result movies and
returns the `top_n` most frequent as `(genre, count)` pairs, most frequent first. A movie counts
once for each of its genres. `rating_facets(results)` sorts the result movies into three bands and
returns a dict `{"<6": ..., "6-8": ..., ">8": ...}` of counts, where `6-8` includes both ends.

In [ ]:
# --- Verify C3 ---
res = engine.search("a romantic love story", pipeline="C", top_k=20)

gf = engine.genre_facets(res, top_n=5)
assert isinstance(gf, list) and len(gf) == 5, "genre_facets returns top_n (genre, count) pairs"
counts = [c for _, c in gf]
assert counts == sorted(counts, reverse=True), "facets must be sorted by count, most frequent first"
assert "Romance" in [g for g, _ in gf], f"a romance query should surface Romance, got {gf}"

rf = engine.rating_facets(res)
assert set(rf) == {"<6", "6-8", ">8"}, "rating_facets has exactly the three bands"
assert sum(rf.values()) == len(res), "every result lands in exactly one band"

# Facets describe whatever set you pass, including a filtered one.
hi = engine.search("a dark crime story", pipeline="C",
                   predicate=lambda m: m["rating"] > 8, top_k=10)
assert engine.rating_facets(hi)[">8"] == len(hi), "all rating>8 hits fall in the >8 band"

print("✓ facets summarise the result set!")

In [ ]:
# --- Explore C3 ---
res = engine.search("a space adventure", pipeline="C", top_k=20)
display_md("**Genre facets for `a space adventure` (top 20):**")
print_table([[g, c] for g, c in engine.genre_facets(res, top_n=6)], headers=["Genre", "Count"])
display_md(f"**Rating bands:** {engine.rating_facets(res)}")
# Compare the facet counts of the top 20 with the facets of the top 100. What shifts, and why?

## Where facets belong

> **How this works:** Write your answer in the markdown cell below the question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you can
> compare and reflect.

### ✏ Task T1 - Turning a clicked facet back into results

Your `genre_facets` and `rating_facets` summarise the current result set. Now the user clicks one,
say the genre "Thriller", to narrow the results. There are two ways to honour that click:

1. Keep only the already-retrieved results whose genre is "Thriller" and redraw the list, touching
   nothing else.
2. Turn the click into a predicate (`lambda m: "Thriller" in m["genres"]`) and run `search` again
   over the whole collection with that predicate.

Answer three questions. First, your facet counts were computed over the top-k results, not the whole
collection. What does that make the counts, and how does it bias what the user sees? Second, compare
the two options above: what does each cost, and when does option 1 return too little? Third, name one
application where option 1 is clearly the right choice, and one where option 2 is necessary.

> *Your answer here...*